In [19]:
# Import section
import os

import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.optim as optim

from music21 import converter, note
from scipy.signal import butter, filtfilt

import matplotlib.pyplot as plt

In [20]:
# Example of bar2_pitches
bar2_pitches = [76.0, 76.0, 76.0, 76.0, 76.0, 76.0, 71.0, 71.0, 71.0, 71.0, 71.0, 71.0, 71.0, 71.0, 71.0, 71.0, 71.0, 71.0, 71.0, 71.0, 71.0, 71.0, 71.0, 71.0, 76.0, 76.0, 76.0, 76.0, 76.0, 71.0, 71.0, 71.0, 71.0, 71.0, 71.0, 79.0, 79.0, 79.0, 79.0, 79.0, 79.0, 71.0, 71.0, 71.0, 71.0, 71.0, 71.0, 71.0]


In [ ]:
# Function for extracting ground truth vectors
def compute_true_vectors(bar2_pitches):
    """
    Obtain Tmidi true and Trp true from the ground truth pitch vector of the 2nd bar

    Args:
        bar2_pitches: [B, 48] of MIDI pitches (0 means rest)

    Returns:
        tmidi_true: [B, 24] tokens (0-127 = notes, 128 = hold, 129 = rest)
        trp_true: [B, 24] quantized relative pitch curve'
    """

    # Define constants from hparams
    batch = 8
    seq_len_dataset = 48
    seq_len = 24

    # Define grid beat ticks
    beat_ticks = [0, 1/4, 1/3, 1/2, 2/3, 3/4] # 6 uneven ticks per beat -> 24 ticks per measure
    #print(len(beat_ticks)) # 6

    # Define quantization bins
    bins = np.array([-9, -5, -1, 0, 1, 5, 9], dtype=np.float32)

    # Define empty vectors
    tmidi_true = torch.zeros(seq_len, dtype=torch.float32)
    trp_true = torch.zeros(seq_len, dtype=torch.float32)
    vchroma_true = torch.zeros(12, dtype=torch.float32)

    # Create new grid
    # - In seq_len_dataset = 48, the grid is even, so 12 steps per beat (48/4)
    # - Map the 6 uneven ticks into 12 uniform steps
    tick_indices = np.zeros_like(beat_ticks)
    for t in range(len(beat_ticks)):
        #print(t)
        tick_indices[t] = round(beat_ticks[t]*12)
    # print(tick_indices)  # = [0, 3, 4, 6, 8, 9]

    # - Create grid
    grid = [] 
    for beat in range(4):
        base = beat * 12
        for tick in tick_indices:
            grid += [int(base + tick)]
    # print(grid) # = [0, 3, 4, 6, 8, 9, 12, 15, 16, 18, 20, 21, 24, 27, 28, 30, 32, 33, 36, 39, 40, 42, 44, 45]

    for i in range(batch):
        pitches = np.array(bar2_pitches) # has seq_len_dataset

        # Get pitch classes
        pitch_classes = np.unique(pitches % 12)

        # Set active classes to 1
        vchroma_true[pitch_classes] = 1.0

        # Map bar2_pitches to uneven 24-positions grid
        pitches = pitches[grid] # has seq_len

        # Compute Tmidi_true
        # Go through the pitch vector of each batch
        for t in range(seq_len):
            #print(t)
            #print(pitches[t])
            #print(tmidi_true)
            if pitches[t] == 0:
                tmidi_true[t] = 129 # rest condition
            elif t>0 and pitches[t] == pitches[t-1] and pitches[t] != 0:
                tmidi_true[t] = 128 # hold condition
            else:
                tmidi_true[t] = pitches[t] # MIDI pitch number for onset position
        #print(tmidi_true.shape) # 24

        # Compute Trp_true
        # Get the onsets vector
        onsets = np.where((tmidi_true >= 0) & (tmidi_true <= 127))[0]
        #print(onsets)

        if len(onsets) >= 2:
            for pos in range(len(onsets) - 1):
                # For every pair of consecutive onsets in positions i and j = i + n where n > 0, 
                # Calculate the MIDI interval Δ = Tmidi[j] - Tmidi[i]
                i = onsets[pos]
                j = onsets[pos + 1]
                delta = tmidi_true[j] - tmidi_true[i]
                n = j - i

                # Then perform a linear interpolation between i + 1 and j (k points), resulting in Trp[k]
                for k in range(i+1, j+1):
                    trp_true[k] = delta * ((k-i)/n)
        #print(trp_true.shape) # 24

        # Quantize Trp_true
        trp_quantizated = torch.zeros(seq_len, dtype=torch.float32)
        bins_tensor = torch.tensor(bins) # convert to tensor so it can be comptatible with trp_true
        for k in range(seq_len):
            trp_quantizated[k] = bins_tensor[np.argmin(np.abs(bins_tensor-trp_true[k]))]
    
    return tmidi_true, trp_quantizated, vchroma_true

In [22]:
#print(bar2_pitches)

tmidi_true, trp_true, vchroma_true = compute_true_vectors(bar2_pitches)
#print(tmidi_true.shape) # 24
#print(trp_true.shape) # 24
#print(vchroma_true.shape) # 12

C:\Users\cristinj\AppData\Local\Temp\ipykernel_28532\2135415829.py:97: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  trp_quantizated[k] = bins_tensor[np.argmin(np.abs(bins_tensor-trp_true[k]))]
